<div style="background-color: black; color: white; padding: 10px;text-align: center;">
  <strong>Date Published:</strong> December 29, 2025 <strong>Author:</strong> Adnan Alaref
</div>

# 🎡 Baseline Fine-Tuning with ResNet50 (8 Classes)

## ✨ Introduction

This notebook presents a **strong and well-established baseline** for group activity recognition using deep learning. Instead of experimenting with unproven ideas, the focus here is on **reliable fine-tuning strategies** that are widely validated in the literature.

Following classical CVPR approaches that relied on **AlexNet**, we adopt a **more powerful backbone (ResNet50)** to benefit from deeper representations and improved generalization.

For each video clip, we initially use **only the middle frame** as a representative image. This simplifies the pipeline while preserving discriminative information. Optionally, temporal context can be incorporated by sampling **5 frames before and 4 frames after** the center frame.

The model is fine-tuned as an **8-class image classifier**, and evaluation metrics are reported to establish a **clean, reproducible first baseline**. This baseline will serve as a reference point for future improvements and more advanced temporal modeling.

> **Note:** The goal of this notebook is not to maximize performance, but to establish a trustworthy and reproducible baseline.
**The ExtendedModel_Baseline1 achieved 75.6% accuracy and a macro F1-score of 0.768, indicating balanced performance across classes and establishing a strong baseline for further improvements.**

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 1: Import Library.</div>

In [ ]:
import re
import os
import cv2
import torch
import numpy as np
import seaborn as sns
import torch.nn as nn
from pathlib import Path
from pprint import pprint
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from typing import Dict, List, TypeVar, Tuple, Any
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.simplefilter(action='ignore')
warnings.filterwarnings(action='ignore', category=FutureWarning)

In [ ]:
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
print("Memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 2: Collect Middle Image From All Videos Directories.</div>

In [ ]:
!ls /kaggle/input/group-activity-recognition-volleyball

In [ ]:
# from google.colab import drive
# drive.mount(r'/content/drive')
dataset_root = r"/kaggle/input/group-activity-recognition-volleyball"

In [ ]:
img =Path(r'/kaggle/input/group-activity-recognition-volleyball/videos/1/9575/9575.jpg')
if not img.exists():
    print(f"Error: File does not exist at {img}")
else:
    image = Image.open(img).convert("RGB")
    plt.imshow(image)
    plt.title(f"Image: {img.name}")
    plt.axis('off')
    plt.show()

In [ ]:
def collect_data(videos_root:str, split_group:List[int])->Dict[str, str]:
  if not os.path.isdir(videos_root):
    raise FileNotFoundError(f"Expected directory does not exist.\nPath: {videos_root}")

  videos_annotations: Dict[str, str] = {}
  for video_id in split_group:
    # Get annotations.txt for each video_id
    video_annot = os.path.join(videos_root, str(video_id), 'annotations.txt')
    if not os.path.isfile(video_annot):
       raise FileNotFoundError(f"Video annotation file does not exist.: \nPath: {video_annot}")

    # Collect data form each annotations file
    with open(video_annot, mode='r') as file:
      for line in file:
        line = line.strip()
        if not line:
          continue # skip empty lines
        parts = line.split()
        if len(parts) < 2:
          continue
        frame_ID, label_group_action = parts[0], parts[1]
        clip_ID = re.sub(r'\.[a-zA-Z0-9]+$','',frame_ID, flags=re.IGNORECASE)
        frame_path = os.path.join(videos_root, str(video_id), clip_ID, frame_ID)
        videos_annotations[frame_path] = label_group_action

  return videos_annotations

In [ ]:
videos_r = os.path.join(dataset_root,'videos')
dic = collect_data(videos_r, [0])
print(type(dic))
print(len(dic.keys()))
Group_actions = set(dic.values())
print(Group_actions)
pprint(dict(list(dic.items())[:3]))

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 3: Create Custom Dateset.</div>

In [ ]:
class Group_activity_Dataset(Dataset):
  def __init__(self, videos_root:str, split:List[int], transform = None) -> None:
    super().__init__()
    self.split = split
    self.transform = transform
    self.videos_root = videos_root

    # Get all data
    videos_annotations = collect_data(self.videos_root, self.split)
    self.samples = list(videos_annotations.items())

    # Create a consistent mapping between action labels and integers for the dataset:
    group_activity_clases = ["r_set", "r_spike", "r-pass", "r_winpoint", "l_winpoint", "l-pass", "l-spike", "l_set"]
    # Map action -> integer converts string labels to integer indices for model training.
    self.Group_action2idx = {action:idx for idx, action in enumerate(group_activity_clases)}
    # Map integer -> action allows converting predicted indices back to string labels.
    self.Group_idx2action = {idx:action for action, idx in self.Group_action2idx.items()}


  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index:int):
    frame_path, group_label = self.samples[index]
    try:
      # frame_path = Path(frame_path)
      image = Image.open(frame_path)
      image = ImageOps.exif_transpose(image) # safe, no-op here
      image = image.convert("RGB")
    except Exception as e:
      raise RuntimeError(f"Failed to load image: {frame_path}") from e

    if self.transform:
      image = self.transform(image=np.array(image))['image']

    group_label = torch.tensor(self.Group_action2idx[group_label], dtype=torch.long)
    return image, group_label

  @property
  def summary(self):
    labels = [g_label for _, g_label in self.samples]
    return {
      "Images_Count": len(self.samples),
      "Unique_Labels": len(set(labels))
    }

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 4: Experiment Configuration.</div>

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class ModelConfig:
  # name: TYPE = VALUE
  dropout: float = 0.5
  num_classes : int = 8
  device: str|None = None
  # mutable → must use field
  train_split:List[int] = field(default_factory=lambda:[
      1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31,
      32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54
      ])

  test_split:List[int] = field(default_factory=lambda:[
      4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47
      ])

  val_split:List[int] = field(default_factory=lambda:[
      0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51
  ])

  classes_names:List[str] = field(default_factory=lambda:[
      "r_set", "r_spike", "r-pass", "r_winpoint", "l_winpoint", "l-pass", "l-spike", "l_set"
  ])

  def __post_init__(self):
    max_index = 54
    if self.device == None:
      self.device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

    # validate splits
    for name, split in {
      "train": self.train_split,
      "test": self.test_split,
      "val": self.val_split,
      }.items():

      # empty
      if len(split) == 0:
        raise ValueError(f"{name}_split cannot be empty.")

      # uniqueness
      if len(split) != len(set(split)):
        raise ValueError(f"{name}_split contains duplicate indices.")

      # range
      if any(idx < 0 or idx > max_index for idx in split):
        raise ValueError(f"{name}_split indices must be in range [0, {max_index}]")

    # data leakage check
    all_index = set(self.train_split) | set(self.val_split) | set(self.test_split)
    if len(all_index) != (len(self.train_split) + len(self.val_split) + len(self.test_split)):
      raise ValueError("Splits overlap, There are data leakage problem!")

@dataclass
class TrainingConfig:
  learning_rate: float = 3e-4
  val_bsize:int = 32
  batch_size: int = 32
  epochs: int = 50
  checkpoint_path: str = r'/kaggle/working/chekpoints'
  save_every: int = 10    # save checkpoint every 10 epochs
  print_every: int = 1
  eta_min: float = 1e-6
  weight_decay: float = 1e-4
  betas:tuple = field(default_factory=lambda:(0.9, 0.999))

model_config = ModelConfig()
train_config = TrainingConfig()

In [ ]:
def save_checkpoints(model: nn.Module, scaler: torch.GradScaler|None,
                     optimizer: torch.optim.Optimizer, train_loss: float, val_loss: float, macro_f1:float, epoch: int, path: str)->None:
  # Define Chekpoint
  checkpoint = {
    "epochs": epoch,
    "macro_f1":float(macro_f1),
    "val_loss": float(val_loss),
    "train_loss": float(train_loss),
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
  }

  # add scaler ONLY if it exists
  if scaler is not None:
    checkpoint['scaler_state_dict'] = scaler.state_dict()

  # Create dir
  os.makedirs(os.path.dirname(path), exist_ok=True)
  torch.save(checkpoint, path)
  print(f"Saved checkpoint: {path}")


def load_checkpoint(path:str, device: str|None=None)->dict:
  if not os.path.isfile(path):
    raise FileNotFoundError(f"Checkpoint file does not found.\nPath: {path}")

  # Load Data
  checkpoint_data = torch.load(path, map_location=device)
  return checkpoint_data

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 5: Dataset and DataLoader Setup.</div>

In [ ]:
!pip install albumentations  >/dev/null 2>&1

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform1 = A.Compose([
    A.Resize(224, 224),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.CoarseDropout(
        max_holes=1, max_height=50, max_width=50, min_holes=1, min_height=50, min_width=50, fill_value=0, p=0.5
    ),
    ToTensorV2()
])

val_transform1 = A.Compose([
    A.Resize(224, 224),
    ToTensorV2()
])

# Path to the videos directory inside the dataset
videos_root = os.path.join(dataset_root,'videos')

# Number of worker processes used for data loading
# > 0 enables parallel data loading
num_workers = os.cpu_count()

# Pin memory is only useful when training on CUDA
# It speeds up CPU → GPU memory transfers
pin_memory = model_config.device == "cuda"

loader_kwargs = dict(
  num_workers = num_workers,
  pin_memory = pin_memory,
  persistent_workers= num_workers > 0,
  # persistent_workers Keeps worker processes alive between epochs
  # Must be False when num_workers == 0
)

# prefetch_factor controls how many batches each worker preloads
# It is only valid when num_workers > 0
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 2

# Training dataset (uses training split indices)
train_dataset = Group_activity_Dataset(videos_root, model_config.train_split, train_transform1)
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=train_config.batch_size,
                          shuffle=True,
                          **loader_kwargs)

# Validation dataset (no data augmentation, no shuffling)
val_dataset = Group_activity_Dataset(videos_root, model_config.val_split, val_transform1)
val_loader = DataLoader(dataset=val_dataset,
                        batch_size=train_config.val_bsize,
                        shuffle=False,
                        **loader_kwargs)

# Test dataset (used only for final evaluation)
test_dataset = Group_activity_Dataset(videos_root, model_config.test_split, val_transform1)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=train_config.val_bsize,
                         shuffle=False,
                         **loader_kwargs)

## <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 5.1: Using Advanced Augmintaion[Mixpu, Cutout].</div>

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """
    x: images [B, C, H, W] (GPU)
    y: labels [B] or [B, num_classes] (GPU)
    """
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 6: Model Architecture Design.</div>

In [ ]:
class ExtendedModel_Baseline1(nn.Module):
  def __init__(self, backbone: nn.Module, num_features:int, num_classes: int, prop_dropout:float) -> None:
    super().__init__()
    self.backbone = backbone
    self.fc_layer = nn.Sequential(
      nn.Dropout(prop_dropout),
      nn.Linear(num_features, num_classes)
    )

  def forward(self, x:torch.Tensor)-> torch.Tensor:
    x = self.backbone(x)    # [B, C, 1, 1] because adativeavgpool is included
    x = torch.flatten(x, 1) # [B, C]
    x = self.fc_layer(x)
    return x

In [ ]:
# Instanciate Model
Weights = models.ResNet50_Weights.DEFAULT
original_model = models.resnet50(weights = Weights)
layers = list(original_model.children())[:-1]  # keep avgpool
truncated_model = nn.Sequential(*layers)

# last_layer = list(truncated_model.children())[-2]
# last_block = last_layer[-1]
# num_last_features = last_block.conv3.out_channels
with torch.no_grad():
  dummy = torch.zeros(1, 3, 224, 224)
  num_last_features = truncated_model(dummy).shape[1]

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
BaseLine1_model = ExtendedModel_Baseline1(truncated_model, num_last_features, model_config.num_classes, model_config.dropout)
optimizer = torch.optim.AdamW(BaseLine1_model.parameters(), lr=train_config.learning_rate, betas=train_config.betas, weight_decay=train_config.weight_decay)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer = optimizer, T_max=train_config.epochs, eta_min=train_config.eta_min)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: (1 - epoch / train_config.epochs))

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 7: Model Architecture Summary.</div>

In [ ]:
try:
  import torchinfo
  print(f"torchinfo version: {torchinfo.__version__}")
except:
  # %%capture
  ! pip install torchinfo > /dev/null 2>&1
  import torchinfo
  print(f"torchinfo version: {torchinfo.__version__}")

dummy_data = torch.zeros(1,3,224,224)
torchinfo.summary(model= BaseLine1_model, input_data=(dummy_data), device = model_config.device)

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 8: Training and Testing Pipeline.</div>

In [ ]:
import random
import numpy as np

def set_all_seeds(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(seed)
      torch.backends.cudnn.deterministic = False
      torch.backends.cudnn.benchmark = True

In [ ]:
def accuracy_fn(y_true, y_pred):
  """Calculates accuracy between truth labels and predictions.

  Args:
      y_true (torch.Tensor): Truth labels for predictions.
      y_pred (torch.Tensor): Predictions to be compared to predictions.

  Returns:
      [torch.float]: Accuracy value between y_true and y_pred, e.g. 78.45
  """
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct / len(y_pred)) * 100
  return acc

In [ ]:
def train_step(model: nn.Module,
               dataloader: torch.utils.data.DataLoader,
               accuracy_fn,
               criterion: torch.nn.Module,
               optimizer : torch.optim.Optimizer,
               scaler: torch.cuda.amp.GradScaler, device:str):

  from torch.cuda.amp import autocast

  total_train_loss, total_train_acc = 0.0, 0.0
  model.to(device)
  model.train()

  for inputs, targets in dataloader:
    inputs = inputs.to(device, non_blocking = True).float()# <-- important
    targets = targets.to(device, non_blocking = True)

    # Add Mixup and Cutout Augmintation correct order coutout -> Mixup
    inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha=0.4)
      
    optimizer.zero_grad(set_to_none=True) # faster & safer

    # Forward and loss in mixed precision
    with autocast():
      logits = model(inputs)
      loss = mixup_criterion(
            criterion, logits, targets_a, targets_b, lam
      )
      train_preds = torch.softmax(logits, dim=1).argmax(dim=1) # Go from logits -> pred labels

    # Backward with scaled loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    total_train_loss += loss.item()
    total_train_acc+= accuracy_fn(y_true=targets, y_pred=train_preds)

  # Average Loss and accuracy for all batches per epoch
  total_train_loss /=len(dataloader)
  total_train_acc /=len(dataloader)

  return {
    "Model_name":model.__class__.__name__,
    "Train_loss":total_train_loss,
    "Train_acc":total_train_acc
  }

In [ ]:
def eval_step(model: nn.Module,
              dataloader: torch.utils.data.DataLoader,
              accuracy_fn, criterion: torch.nn.Module, device:str):
  """Returns a dictionary containing the results of model predicting on data_loader.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        dataloader (torch.utils.data.DataLoader): The target dataset to predict on.
        criterion (torch.nn.Module): The loss function of model.
        accuracy_fn: An accuracy function to compare the models predictions to the truth labels.
        device: which device we use it.

    Returns:
        (dict): Results of model making predictions on dataloader.
  """
  from torch.cuda.amp import autocast
    
  model.to(device)
  model.eval()
    
  all_targets, all_preds = [], []
  total_test_loss, total_test_acc = 0.0, 0.0

  with torch.inference_mode():
    for inputs, targets in dataloader:
      inputs = inputs.to(device, non_blocking = True).float()# <-- important
      targets = targets.to(device, non_blocking = True)

      # Mixed precision inference
      with autocast():
        test_logits = model(inputs)
        test_loss = criterion(test_logits, targets)
        test_preds = test_logits.argmax(dim=1) # Go from logits -> pred labels

      total_test_loss +=test_loss.item()
      total_test_acc +=accuracy_fn(y_true=targets, y_pred=test_preds)
        
      all_preds.append(test_preds.cpu())
      all_targets.append(targets.cpu())

  # Average Loss and accuracy for all batches per epoch
  total_test_loss /=len(dataloader)
  total_test_acc /=len(dataloader)

  y_preds = torch.cat(all_preds)
  y_true = torch.cat(all_targets)
  macro_f1 = f1_score(y_true, y_preds, average='macro')

  return {
    "Model_name":model.__class__.__name__,
    "Test_loss":total_test_loss,
    "Test_acc":total_test_acc,
    "Macro_f1": macro_f1
  }

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 9: Training and Testing Model With Mixed Precision.</div>

In [ ]:
from torch.cuda.amp import  GradScaler # PERFORMANCE OPTIMIZATIONS
scaler = GradScaler()  # helps scale gradients safely

set_all_seeds(42)
train_history, val_history = {}, {}
epoch_values, val_losses_values, train_losses_values, lr_history = [], [], [], []

best_loss = float('inf')
best_macro_f1 = 0.0
for epoch in range(train_config.epochs):

  # ---- Train ----
  train_history = train_step(model= BaseLine1_model,
                             dataloader= train_loader,
                             accuracy_fn= accuracy_fn,
                             criterion= criterion,
                             optimizer= optimizer,
                             scaler= scaler,
                             device= model_config.device)

  # ---- Validate ----
  val_history = eval_step(model= BaseLine1_model,
                          dataloader= val_loader,
                          accuracy_fn= accuracy_fn,
                          criterion= criterion,
                          device= model_config.device)


  # ---- Logging ----
  epoch_values.append(epoch)
  val_losses_values.append(val_history['Test_loss'])
  lr_history.append(optimizer.param_groups[0]['lr'])
  train_losses_values.append(train_history['Train_loss'])

  # ---- Checkpoint ----
  if val_history['Macro_f1'] > best_macro_f1:
    best_macro_f1 = val_history['Macro_f1']
    save_checkpoints(BaseLine1_model, scaler, optimizer, train_history['Train_loss'], val_history['Test_loss'], best_macro_f1, epoch+1, os.path.join(train_config.checkpoint_path, f"checkpoint_epoch{epoch+1}.pt"))

  # ---------- SCHEDULER ----------
  # CosineAnnealingLR
  scheduler.step() # This scheduler check per epoch or batch,  ALWAYS after validation
    
  # ---- Print ----
  if(epoch+1) % train_config.print_every == 0:
    print(f"Epoch [{epoch+1:>2}] - Macro F1: {val_history['Macro_f1']:.3f} | Train loss: {train_history['Train_loss']:.3f} | Train acc: {train_history['Train_acc']:.3f}% | Test loss: {val_history['Test_loss']:.3f} | Test acc: {val_history['Test_acc']:.3f}% | LR: {scheduler.get_last_lr()[0]:.3f}")

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 10: Evaluation Model.</div>

In [ ]:
model = ExtendedModel_Baseline1(
    backbone= truncated_model,
    num_features= num_last_features,
    num_classes= model_config.num_classes,
    prop_dropout= model_config.dropout
)

res = load_checkpoint(path = r'/kaggle/working/chekpoints/checkpoint_epoch41.pt',device= model_config.device)
# res.keys()
model.load_state_dict(res['model_state_dict'])

In [ ]:
Model_result = eval_step(model, test_loader, accuracy_fn, criterion, model_config.device)
Model_result

In [ ]:
model.to(model_config.device)
model.eval()

y_true, y_preds = [], []
model.eval()
with torch.inference_mode():
    for X, y in test_loader:
        X = X.to(model_config.device, non_blocking=True).float()# <-- important
        y = y.to(model_config.device, non_blocking=True)

        logits = model(X)
        preds = logits.argmax(dim=1)

        y_preds.append(preds.cpu())
        y_true.append(y.cpu())

y_preds_tensor = torch.cat(y_preds)
y_true_tensor  = torch.cat(y_true)

In [ ]:
# Let's plot Confusion Matrix by Using mlxtend
# 1- See if torchmetrics exists, if not, install it
try:
  import torchmetrics, mlxtend
  print(f"mlxtend version: {mlxtend.__version__}")
  assert int(mlxtend.__version__.split(".")[1])>=19,  "mlxtend verison should be 0.19.0 or higher"
except:
  # %%capture
  ! pip install torchmetrics > /dev/null 2>&1 && pip install -U mlxtend >/dev/null 2>&1
  import torchmetrics, mlxtend
  print(f"mlxtend version: {mlxtend.__version__}")

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix


# 1. Setup confusion matrix instance and compare predictions to targets
confmat = ConfusionMatrix(task="multiclass", num_classes=len(model_config.classes_names))
confmat_tensor = confmat(preds = y_preds_tensor, target=y_true_tensor)

# 3. Plot the confusion matrix
fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.cpu().numpy(), # matplotlib likes working with NumPy
                            class_names = model_config.classes_names,
                            figsize=(10,7))
plt.tight_layout()
plt.show()

In [ ]:
def Display_train_test_loss_curve(epochs,train_losses,test_losses):
  plt.figure(figsize=(6,5) ,dpi=100)
  plt.plot(epochs,train_losses,label = "Train Loss")
  plt.plot(epochs ,test_losses, label = "Test Loss")
  plt.ylabel("Loss" ,fontsize = 12 ,weight='bold')
  plt.xlabel("Epochs" ,fontsize = 12 ,weight='bold')
  plt.title(f"Traning and Testing Loss Curves" ,fontsize = 12 ,weight='bold')
  plt.legend(prop = {"size":14},loc = "best")
  plt.grid(True)

  plt.tight_layout()
  # plt.axis("off")
  plt.show()

Display_train_test_loss_curve(epoch_values,train_losses_values,val_losses_values)

In [ ]:
import ipywidgets as widgets
widgets.Widget.widget_types

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Thanks & Upvote ❤️</div>